In python, the standard way to use transformer models it to work with the Transformers library, that offers access to pre-trained language models and a user-friendly pipeline for different tasks, including text classification.

Check out the  [Transformer library](https://huggingface.co/docs/transformers/index) for more details.

Browse the repository of pre-trained models for text classification [here](https://huggingface.co/models?pipeline_tag=text-classification&sort=trending)

<br>
<a target="_blank" href="https://colab.research.google.com/drive/10d88csqxt7ClGnVvmwrJmfoGL_5u_RQB#scrollTo=66mf1h_5JKga">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

By default, Colab will run on CPUs. If you want to use GPU hardware and accelerate the computational process, you can go on Runtime, change runtime type on GPU and then Click on Connect on the upper right of the interface.

---



In [ ]:
%%capture

# Equivalent to install.packages("") in R. But here, you have to install them everytime.

!pip install transformers==4.44.1 # The main library for accessing transformers from Huggingface
!pip install langdetect # A library to use language detection


The Transformers library provides a simple tool called a **pipeline** that significantly simplifies the process of using existing pre-trained language models for various tasks. It abstracts away the complexity of model loading, tokenization, and inference, allowing you to easily apply models to tasks like text classification, question answering, translation, and masked language modeling. By calling a pipeline with a specific task, such as "text-classification," you can quickly get predictions with minimal code. In this guide, we will explore different examples to demonstrate how to load a model through a pipeline. Each time, the pipeline requires you to specify:

- The task you want to perform
- The model you want to use from Hugging Face


In [ ]:
# Set up GPU

import torch

# use GPU (cuda) if available, otherwise use CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")


In [ ]:
from transformers import pipeline # Equivalent to library() in R



## Mask language modelling

Masked Language Modeling (MLM) is a technique where a model is trained to predict missing words in a sentence by replacing random words with a special token (e.g., [MASK]). For example, in the sentence, "The cat sat on the [MASK]," the model predicts "mat" based on the context.

MLM forms the basis for pre-trained models like BERT, which learn rich, contextual representations of words. This enables the models to understand word meanings in various contexts, making them effective for tasks like text classification, question answering, and sentiment analysis.

These models are also key to transfer learning. After being pre-trained with MLM, they can be fine-tuned on specific tasks, saving time and resources. This makes them highly adaptable for a wide range of NLP applications, such as named entity recognition, text classification, and more.









In [ ]:
# Initialize the fill-mask pipeline with a pre-trained model (e.g., BERT)
fill_mask = pipeline("fill-mask", model="bert-base-uncased")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


In [ ]:
# Example sentences with a [MASK] token
sentences = [
    "The [MASK] of the Arctic region is changing rapidly, with temperatures rising at an alarming rate.",
    "The political [MASK] has been tense in the country, with protests growing in various cities over the past few months."
]

# Use the pipeline to predict the masked word for both sentences
for sentence in sentences:
    predictions = fill_mask(sentence)
    print(f"Predictions for sentence: {sentence}")

    # Print the top 5 predictions for the masked token
    for prediction in predictions:
        print(f"Prediction: {prediction['token_str']} - Score: {prediction['score']}")
    print("\n")  # Print a newline between the results for readability

Predictions for sentence: The [MASK] of the Arctic region is changing rapidly, with temperatures rising at an alarming rate.
Prediction: climate - Score: 0.9516469240188599
Prediction: temperature - Score: 0.02384592592716217
Prediction: geography - Score: 0.0045017520897090435
Prediction: landscape - Score: 0.003932666499167681
Prediction: weather - Score: 0.0033565587364137173


Predictions for sentence: The political [MASK] has been tense in the country, with protests growing in various cities over the past few months.
Prediction: situation - Score: 0.8717061877250671
Prediction: climate - Score: 0.09481281787157059
Prediction: environment - Score: 0.008607661351561546
Prediction: atmosphere - Score: 0.008271207101643085
Prediction: scene - Score: 0.0050366781651973724




Similar to static embeddings, you can use these contextual representations to learn about model biases based on how likely a given word would be predict by a model in different sentences. Here below an example taken from [Hauke Licht's course on Transformers](https://github.com/haukelicht/advanced_text_analysis/blob/main/notebooks/contextualized_embedding_transformers_explained.ipynb).


In [ ]:
# Example sentences with a [MASK] token
sentences = [
    "Homosexuals are making our country [MASK].",
    "Straights are making our country [MASK]."
]
# Use the pipeline to predict the masked word for both sentences
for sentence in sentences:
    predictions = fill_mask(sentence)
    print(f"Predictions for sentence: {sentence}")

    # Print the top 5 predictions for the masked token
    for prediction in predictions:
        print(f"Prediction: {prediction['token_str']} - Score: {prediction['score']}")
    print("\n")  # Print a newline between the results for readability

Predictions for sentence: Homosexuals are making our country [MASK].
Prediction: worse - Score: 0.14274117350578308
Prediction: miserable - Score: 0.05214320495724678
Prediction: unsafe - Score: 0.04907706379890442
Prediction: safer - Score: 0.03933172672986984
Prediction: dangerous - Score: 0.038073983043432236


Predictions for sentence: Straights are making our country [MASK].
Prediction: proud - Score: 0.1590529978275299
Prediction: better - Score: 0.0672646015882492
Prediction: stronger - Score: 0.06301791220903397
Prediction: miserable - Score: 0.03387586772441864
Prediction: safer - Score: 0.029428131878376007




In [ ]:
# Example sentences with a [MASK] token
sentences = [
    "Men should [MASK].",
    "Women should [MASK]."
]
# Use the pipeline to predict the masked word for both sentences
for sentence in sentences:
    predictions = fill_mask(sentence)
    print(f"Predictions for sentence: {sentence}")

    # Print the top 5 predictions for the masked token
    for prediction in predictions:
        print(f"Prediction: {prediction['token_str']} - Score: {prediction['score']}")
    print("\n")  # Print a newline between the results for readability

Predictions for sentence: Men should [MASK].
Prediction: fight - Score: 0.0709475725889206
Prediction: die - Score: 0.06575854867696762
Prediction: know - Score: 0.04549961909651756
Prediction: talk - Score: 0.031104378402233124
Prediction: be - Score: 0.02514948882162571


Predictions for sentence: Women should [MASK].
Prediction: know - Score: 0.10045552998781204
Prediction: be - Score: 0.05500389635562897
Prediction: understand - Score: 0.040554508566856384
Prediction: talk - Score: 0.03237244114279747
Prediction: work - Score: 0.027062635868787766




## Text classification

As we have already seen earlier in the course, text classification is one of the most common tasks in text analysis, with many potential applications in political science. There are plenty of pre-trained models in Hugging Face that allow you to classify a wide variety of things. Most of these models are models that use the versions of BERT and fine-tune it to specific classification tasks.

### Sentiment

Rather than using more classical sentiment analysis through dictionaries, it is possible to use existing sentiment classifiers trained for this tasK

In [ ]:
sentiment_classifier = pipeline("sentiment-analysis",model="cardiffnlp/twitter-roberta-base-sentiment-latest")

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


In [ ]:
sentiment_tweets = [
    "We're proud to announce our new green energy policy to create a sustainable future for all! 🌱 #GoGreen #FutureFirst",
    "The opposition party's plan lacks substance and is not in the best interest of the public. #PolicyFailure",
    "Join us this weekend for our community outreach program. Let's work together to make a difference. #CommunityFirst",
    "Recent reports show that our healthcare reforms are making a difference, but there's still work to do. #Progress",
    "Shocking incompetence from the government on handling the cost of living crisis. We deserve better! #LeadershipFail",
    "Our vision for the future: a stronger economy, better jobs, and opportunities for everyone. #TogetherWeCan",
    "Debate tonight at 8 PM. Tune in to hear our plans for the next generation. #Election2025",
]

# Classify emotions and get the results with scores
sentiment_predictions = [sentiment_classifier(text) for text in sentiment_tweets]

# Print the text with corresponding emotion and scores
for text, result in zip(sentiment_tweets, sentiment_predictions):
    print(f"Text: {text}")
    for sentiment in result:
        print(f"Sentiment: {sentiment['label']}, Score: {sentiment['score']:.4f}")
    print("-----")

Text: We're proud to announce our new green energy policy to create a sustainable future for all! 🌱 #GoGreen #FutureFirst
Sentiment: positive, Score: 0.9838
-----
Text: The opposition party's plan lacks substance and is not in the best interest of the public. #PolicyFailure
Sentiment: negative, Score: 0.9035
-----
Text: Join us this weekend for our community outreach program. Let's work together to make a difference. #CommunityFirst
Sentiment: positive, Score: 0.9177
-----
Text: Recent reports show that our healthcare reforms are making a difference, but there's still work to do. #Progress
Sentiment: positive, Score: 0.6701
-----
Text: Shocking incompetence from the government on handling the cost of living crisis. We deserve better! #LeadershipFail
Sentiment: negative, Score: 0.9492
-----
Text: Our vision for the future: a stronger economy, better jobs, and opportunities for everyone. #TogetherWeCan
Sentiment: positive, Score: 0.9657
-----
Text: Debate tonight at 8 PM. Tune in to hear

### Emotions

In [ ]:
# Use a text classification model to detect emotions in text

emotion_classifier = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base", trust_remote_code=True)


emotion_texts = [
    "The government's failure to address climate change is a betrayal to future generations.",
    "I'm incredibly proud of the progress we’ve made in passing comprehensive healthcare reform.",
    "How can politicians sleep at night knowing how many people are suffering because of their policies?",
    "I can't believe the corruption in the system. It’s become so blatant and nothing ever gets done about it.",
    "The election results were shocking, and many of us are still trying to process everything.",
    "We need urgent action to tackle poverty, but all we see are empty promises and no real solutions.",
    "It's inspiring to see so many young people getting involved in politics and pushing for real change.",
    "I’m deeply concerned about the growing divide in our country. We need to find a way to unite again."
]

# Classify emotions and get the results with scores
emotion_predictions = [emotion_classifier(text) for text in emotion_texts]

# Print the text with corresponding emotion and scores
for text, result in zip(emotion_texts, emotion_predictions):
    print(f"Text: {text}")
    for emotion in result:
        print(f"Emotion: {emotion['label']}, Score: {emotion['score']:.4f}")
    print("-----")


config.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/329M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/294 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


Text: The government's failure to address climate change is a betrayal to future generations.
Emotion: disgust, Score: 0.6021
-----
Text: I'm incredibly proud of the progress we’ve made in passing comprehensive healthcare reform.
Emotion: joy, Score: 0.7973
-----
Text: How can politicians sleep at night knowing how many people are suffering because of their policies?
Emotion: neutral, Score: 0.3800
-----
Text: I can't believe the corruption in the system. It’s become so blatant and nothing ever gets done about it.
Emotion: disgust, Score: 0.7124
-----
Text: The election results were shocking, and many of us are still trying to process everything.
Emotion: surprise, Score: 0.9210
-----
Text: We need urgent action to tackle poverty, but all we see are empty promises and no real solutions.
Emotion: sadness, Score: 0.6377
-----
Text: It's inspiring to see so many young people getting involved in politics and pushing for real change.
Emotion: joy, Score: 0.7624
-----
Text: I’m deeply concer

## Topic classification

If you are looking at political texts and want to classify them according policy issues, there are different models available that automate the classification of the [Comparative Agendas Project](https://www.comparativeagendas.net/) into ~20 policy issues. For instance, the  [poltext lab repository](https://huggingface.co/poltextlab) contains more than 80 different models fine-tuned on CAP data of different languages and types of political texts.



In [ ]:
# Use a topic classification model []

topic_classifier = pipeline("text-classification", model = "cornelius/partypress-multilingual", tokenizer = "cornelius/partypress-multilingual")

config.json:   0%|          | 0.00/2.07k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/712M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/335 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.92M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


In [ ]:
topic_texts =  [
    "Addressing the climate crisis demands bold action.",
    "Governments must prioritize investments in renewable energy.",
    "Universal access to affordable healthcare is a moral imperative.",
    "Expanding public health insurance options can save lives.",
    "Education reform must focus on equity and accessibility for all.",
    "Reducing wealth inequality requires progressive taxation policies.",
    "Immigration policies should balance security with compassion.",
    "Investments in infrastructure can drive economic growth and resilience.",
    "We should push for more federalism at the european level"
    "A strong democracy depends on the protection of voting rights.",
    "Addressing systemic racism is essential for social justice.",
    "Affordable housing policies can reduce homelessness.",
    'We disagree with Trump',
    "Supporting small businesses fosters local economic growth.",
    "Foreign policy should prioritize diplomacy and multilateral cooperation."
]


# Classify each text and store results
classified_results = [
    {"text": text, **topic_classifier(text)[0]}
    for text in topic_texts
]

# Print the results
for result in classified_results:
    print(f"Text: {result['text']}")
    print(f"Topic: {result['label']}")
    print(f"Score: {result['score']:.4f}")
    print("-" * 40)


Text: Addressing the climate crisis demands bold action.
Topic: 7 - Environment
Score: 0.9803
----------------------------------------
Text: Governments must prioritize investments in renewable energy.
Topic: 8 - Energy
Score: 0.9814
----------------------------------------
Text: Universal access to affordable healthcare is a moral imperative.
Topic: 3 - Health
Score: 0.9401
----------------------------------------
Text: Expanding public health insurance options can save lives.
Topic: 3 - Health
Score: 0.4718
----------------------------------------
Text: Education reform must focus on equity and accessibility for all.
Topic: 6 - Education
Score: 0.9918
----------------------------------------
Text: Reducing wealth inequality requires progressive taxation policies.
Topic: 1 - Macroeconomics
Score: 0.8376
----------------------------------------
Text: Immigration policies should balance security with compassion.
Topic: 9 - Immigration
Score: 0.9891
--------------------------------------

In [ ]:
immigration_multi_sentences = [
    "Immigration is a major topic in today's political discussions.",  # English
    "L'immigration est un sujet majeur dans les discussions politiques actuelles.",  # French
    "Die Einwanderung ist ein wichtiges Thema in den heutigen politischen Diskussionen.",  # German
    "La inmigración es un tema importante en los debates políticos actuales.",  # Spanish
    "L'immigrazione è un tema importante nei dibattiti politici odierni.",  # Italian
    "A imigração é um tema importante nas discussões políticas atuais.",  # Portuguese
    "La inmigración es un tema clave en los debates políticos actuales.",  # Catalan
    "İmmigrasyon, bugünün politik tartışmalarında önemli bir konudur.",  # Turkish
    "Emigracija je tema, o kojoj se danas često diskutuje u političkim krugovima."  # Serbian
]

# Classify each text and store results
multilingual_results = [
    {"text": text, **topic_classifier(text)[0]}
    for text in immigration_multi_sentences
]

# Print the results
for result in multilingual_results:
    print(f"Text: {result['text']}")
    print(f"Topic: {result['label']}")
    print(f"Score: {result['score']:.4f}")
    print("-" * 40)


Text: Immigration is a major topic in today's political discussions.
Topic: 9 - Immigration
Score: 0.8763
----------------------------------------
Text: L'immigration est un sujet majeur dans les discussions politiques actuelles.
Topic: 9 - Immigration
Score: 0.9874
----------------------------------------
Text: Die Einwanderung ist ein wichtiges Thema in den heutigen politischen Diskussionen.
Topic: 9 - Immigration
Score: 0.6355
----------------------------------------
Text: La inmigración es un tema importante en los debates políticos actuales.
Topic: 9 - Immigration
Score: 0.6872
----------------------------------------
Text: L'immigrazione è un tema importante nei dibattiti politici odierni.
Topic: 9 - Immigration
Score: 0.9790
----------------------------------------
Text: A imigração é um tema importante nas discussões políticas atuais.
Topic: 98 - Non-thematic
Score: 0.9301
----------------------------------------
Text: La inmigración es un tema clave en los debates políticos ac

It is of course possible to run this models on a larger number of texts.

In [ ]:
import pandas as pd

politique_generale = pd.read_csv("https://raw.githubusercontent.com/luissattelmayer/intro-css/refs/heads/main/data/politique_generale.csv")
politique_generale


,text,date,intervenant
0,"Madame la Présidente, d'abord le Gouvernement,...",2025-01-14,François Bayrou
1,"Eh bien, Monsieur le Président, Mesdames et Me...",2024-10-02,Michel Barnier
2,Mme la présidente\nL'ordre du jour appelle la ...,2024-10-01,Michel Barnier
3,"Monsieur le Président,\nMesdames et Messieurs ...",2024-01-31,Gabriel Attal
4,"Madame la Présidente,\nMesdames et Messieurs l...",2024-01-30,Gabriel Attal
5,Gérard LARCHER\nVous avez donc la parole.\nÉli...,2022-07-06,Élisabeth Borne
6,"Madame la présidente,\nMesdames et messieurs l...",2022-07-06,Élisabeth Borne
7,"Mesdames et Messieurs les députés,\n\nC'est un...",2020-07-15,Jean Castex
8,"Monsieur le Président,\nMesdames et Messieurs ...",2017-07-04,Édouard Philippe
9,"Monsieur le Président, \nMesdames et Messieurs...",2016-12-13,Bernard Cazeneuve


In [ ]:

import nltk
from nltk.tokenize import sent_tokenize
nltk.download('punkt')
nltk.download('punkt_tab')

politique_generale["sentences"] = politique_generale["text"].apply(sent_tokenize)

# Epxlode the sentences into rows

politique_generale_sent = politique_generale.explode("sentences")
politique_generale_sent

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


,text,date,intervenant,sentences
0,"Madame la Présidente, d'abord le Gouvernement,...",2025-01-14,François Bayrou,"Madame la Présidente, d'abord le Gouvernement,..."
0,"Madame la Présidente, d'abord le Gouvernement,...",2025-01-14,François Bayrou,"Mesdames et Messieurs les députés, en vérité, ..."
0,"Madame la Présidente, d'abord le Gouvernement,...",2025-01-14,François Bayrou,"Sur ces bancs, même parmi ceux qui sont violem..."
0,"Madame la Présidente, d'abord le Gouvernement,...",2025-01-14,François Bayrou,"Et 84 % des Français jugent, paraît-il, que le..."
0,"Madame la Présidente, d'abord le Gouvernement,...",2025-01-14,François Bayrou,Et il m'arrive même de me demander où les 16 %...
...,...,...,...,...
33,Assurer la dignité et la liberté de la personn...,1959-01-15,Michel Debré,"Cependant, me semble-t-il, au milieu des diffi..."
33,Assurer la dignité et la liberté de la personn...,1959-01-15,Michel Debré,"L'autorité du chef de l'État, le souvenir des ..."
33,Assurer la dignité et la liberté de la personn...,1959-01-15,Michel Debré,"Nous devons, mais nous pouvons aussi donner à ..."
33,Assurer la dignité et la liberté de la personn...,1959-01-15,Michel Debré,"C'est, en fin de compte, Mesdames, Messieurs l..."


In [ ]:
from tqdm import tqdm # Import library to have progress bars

def apply_model_to_text(df, text_column, classifier):
    # Initialize tqdm for progress bar
    tqdm.pandas()  # This allows the progress_apply method to work

    # Apply the classifier to the text column and get the result as a DataFrame with a progress bar
    results = df[text_column].progress_apply(lambda x: classifier(x)[0])  # Apply model on each text

    # Create new columns for the label and score
    df['label'] = results.apply(lambda x: x['label'])  # Extract the label
    df['score'] = results.apply(lambda x: x['score'])  # Extract the score

    return df

# Example usage
df_classified = apply_model_to_text(politique_generale_sent, 'sentences', topic_classifier)

df_classified



  1%|          | 97/13316 [00:15<36:15,  6.08it/s]


KeyboardInterrupt: 

In [ ]:
df_classified.value_counts("label")

NameError: name 'df_classified' is not defined

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df_classified['date'] = pd.to_datetime(df_classified['date'])

# Count total sentences per date
total_counts = df_classified.groupby('date').size().reset_index(name='total_count')

# Count immigration sentences per date
immigration_counts = df_classified[df_classified['label'] == '9 - Immigration'] \
    .groupby('date').size().reset_index(name='immigration_count')

# Merge total and immigration counts on date
merged_counts = pd.merge(total_counts, immigration_counts, on='date', how='left')

# Fill NaN values for dates without immigration mentions
merged_counts['immigration_count'] = merged_counts['immigration_count'].fillna(0)

# Compute the share of immigration
merged_counts['immigration_share'] = merged_counts['immigration_count'] / merged_counts['total_count']

# Plot the share of immigration over time
plt.figure(figsize=(12, 6))
plt.plot(merged_counts['date'], merged_counts['immigration_share'], marker='o', linestyle='-')
plt.title("Share of Immigration Mentions Over Time", fontsize=16)
plt.xlabel("Date", fontsize=14)
plt.ylabel("Share of Immigration Mentions", fontsize=14)
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


NameError: name 'df_classified' is not defined

## Sexism

There are also plenty of other classifiers that you might play with. Here for instance, I found a classifier fine-tuned to detect sexism.

In [ ]:
sexism_classifier = pipeline("text-classification", model = "NLP-LTU/distilbert-sexism-detector")

# List of texts
texts_sexism = [
    "Women should stay at home and take care of the children, not pursue careers.",
    "Men are naturally better leaders than women because of their strength and decisiveness.",
    "Everyone should have the opportunity to pursue their dreams, regardless of gender.",
    "It’s important to encourage girls and boys equally to pursue careers in STEM fields.",
    "Girls are just not good at math or science; that’s why they don’t excel in these subjects."
]


# Classify the texts
results = sexism_classifier(texts_sexism)

# Print the text with its corresponding label
for text, result in zip(texts_sexism, results):
    print(f"Text: {text}")
    print(f"Label: {result['label']}")
    print("-----")


config.json:   0%|          | 0.00/750 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/360 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


Text: Women should stay at home and take care of the children, not pursue careers.
Label: sexist
-----
Text: Men are naturally better leaders than women because of their strength and decisiveness.
Label: sexist
-----
Text: Everyone should have the opportunity to pursue their dreams, regardless of gender.
Label: not sexist
-----
Text: It’s important to encourage girls and boys equally to pursue careers in STEM fields.
Label: not sexist
-----
Text: Girls are just not good at math or science; that’s why they don’t excel in these subjects.
Label: sexist
-----


## Zero-shot classification

In [ ]:
# Use a zero-shot model

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


zero_shot_classifier = pipeline("zero-shot-classification", model="mlburnham/Political_DEBATE_base_v1.0", device = device) # To use the base model


hypothesis_template = 'The author of this text is {}.'
test_labels = ['sexist', 'not sexist']

zero_shot_classifier(texts_sexism, test_labels, hypothesis_template = hypothesis_template, multi_label = True)


[{'sequence': 'Women should stay at home and take care of the children, not pursue careers.',
  'labels': ['sexist', 'not sexist'],
  'scores': [0.0003425590693950653, 3.9228831155924127e-05]},
 {'sequence': 'Men are naturally better leaders than women because of their strength and decisiveness.',
  'labels': ['sexist', 'not sexist'],
  'scores': [0.9943988919258118, 0.0036623424384742975]},
 {'sequence': 'Everyone should have the opportunity to pursue their dreams, regardless of gender.',
  'labels': ['not sexist', 'sexist'],
  'scores': [0.996089518070221, 1.6679909094818868e-05]},
 {'sequence': 'It’s important to encourage girls and boys equally to pursue careers in STEM fields.',
  'labels': ['not sexist', 'sexist'],
  'scores': [0.999641478061676, 1.926485765579855e-06]},
 {'sequence': 'Girls are just not good at math or science; that’s why they don’t excel in these subjects.',
  'labels': ['sexist', 'not sexist'],
  'scores': [0.9564207792282104, 2.3587651867273962e-06]}]

## Token classification

In [ ]:
ner_classifier = pipeline("token-classification", model="dslim/bert-base-NER", grouped_entities=True)

sentence = "Barack Obama, the 44th President of the United States, was born in Honolulu, Hawaii, on August 4, 1961, and graduated from Harvard Law School."

ner_classifier(sentence)

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.
/usr/local/lib/python3.11/dist-packages/transformers/pipelines/token_classification.py:168: UserWarning: `grouped_entities` is deprecated and will be removed in version v5.0.0, defaulted to `aggregation_strategy="AggregationStrategy.SIMPLE"` instead.
  warnings.warn(


[{'entity_group': 'PER',
  'score': 0.99947,
  'word': 'Barack Obama',
  'start': 0,
  'end': 12},
 {'entity_group': 'LOC',
  'score': 0.9993695,
  'word': 'United States',
  'start': 40,
  'end': 53},
 {'entity_group': 'LOC',
  'score': 0.9991341,
  'word': 'Honolulu',
  'start': 67,
  'end': 75},
 {'entity_group': 'LOC',
  'score': 0.99967897,
  'word': 'Hawaii',
  'start': 77,
  'end': 83},
 {'entity_group': 'ORG',
  'score': 0.98610276,
  'word': 'Harvard Law School',
  'start': 123,
  'end': 141}]

## Generative models (Decoders)

In [ ]:
from transformers import set_seed
from transformers import pipeline

generator = pipeline('text-generation', model='deepseek-ai/DeepSeek-R1-Distill-Qwen-7B')
set_seed(42)

generator("Donald Trump is", max_length=30, num_return_sequences=5)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/680 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/28.1k [00:00<?, ?B/s]

model-00001-of-000002.safetensors:   0%|          | 0.00/8.61G [00:00<?, ?B/s]

model-00002-of-000002.safetensors:   0%|          | 0.00/6.62G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

## Translation

You can access various open and free translation models that allow you to translate large amounts of text without needing to pay for services like DeepL or Google Translate. This is particularly useful if you're training a multilingual model and need to annotate texts in different languages. For example, you can translate a sample of these texts into a single target language, such as English, while still using the original texts to train the model.

Open-source machine translation (MT) models enable translation between multiple languages without relying on commercial services. The University of Helsinki has uploaded models for over 1,000 language pairs to the Hugging Face hub, and Facebook AI has open-sourced several multilingual models. The EasyNMT library provides a simple wrapper for these models. While most machine translation models translate between two languages in one direction (e.g., German to English, but not the reverse), some are capable of handling translations in multiple directions.


In [ ]:
from langdetect import detect  # Importing langdetect for automatic language detection

# Initialize the translation pipeline with a pre-trained model
pipeline_translate = pipeline("translation", model="facebook/m2m100_418M")

config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.94G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/298 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/3.71M [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.14k [00:00<?, ?B/s]

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


In [ ]:
# Example sentences in different languages
texts = [
    "Climate change is causing more frequent and intense storms.",
    "El cambio climático está causando tormentas más frecuentes e intensas.",
    "Le changement climatique provoque des tempêtes plus fréquentes et plus intenses.",
    "Der Klimawandel verursacht häufigere und intensivere Stürme.",
    "Il cambiamento climatico sta causando tempeste più frequenti e intense.",
    "As mudanças climáticas estão causando tempestades mais frequentes e intensas.",
    "Klimaatverandering veroorzaakt frequentere en intensere stormen.",
    "تغير المناخ يسبب عواصف أكثر تواترًا وشدة."
]


# Translate the sentences into English
for text in texts:
    # Detect the source language using langdetect
    src_lang = detect(text)

    # Translate the sentence
    translated_text = pipeline_translate(text, src_lang=src_lang, tgt_lang="en")[0]['translation_text']

    # Print the detected language and the translation
    print(f"Detected Language: {src_lang}")
    print(f"Original: {text}")
    print(f"Translated: {translated_text}")
    print()  # Print a newline for readability


Detected Language: en
Original: Climate change is causing more frequent and intense storms.
Translated: Climate change is causing more frequent and intense storms.

Detected Language: es
Original: El cambio climático está causando tormentas más frecuentes e intensas.
Translated: Climate change is causing more frequent and intense storms.

Detected Language: fr
Original: Le changement climatique provoque des tempêtes plus fréquentes et plus intenses.
Translated: Climate change causes more frequent and intense storms.

Detected Language: de
Original: Der Klimawandel verursacht häufigere und intensivere Stürme.
Translated: Climate change causes more frequent and intense storms.

Detected Language: it
Original: Il cambiamento climatico sta causando tempeste più frequenti e intense.
Translated: Climate change is causing more frequent and intense storms.

Detected Language: pt
Original: As mudanças climáticas estão causando tempestades mais frequentes e intensas.
Translated: Climate change i

In [ ]:
# Initialize the summarization pipeline with a pre-trained model (e.g., T5)
summarizer = pipeline("summarization", model="t5-small")


# Provided text
text = """
The end of the cessation of hostilities in Gaza is deeply concerning, I urge all sides not to squander progress made over the last week.
All sides must work for a return to cessation that would allow for the release of more hostages, provide much needed time and space to tackle the humanitarian crisis in Gaza, and open a dialogue for a political solution that provides for a long-term cessation of hostilities.
We will only reach that long-term solution if Israel is assured that Hamas cannot carry out an attack like October 7 ever again. Those who can influence Hamas must demand they release the remaining hostages immediately.
The levels of death and destruction over the past weeks has been intolerable. Far too many innocent Palestinians, including women and children, have been killed as part of military operations. There must be full accountability for all actions.
As fighting sadly resumes, Israel must not besiege or blockade Gaza. They must comply with international law by protecting innocent lives and civilian infrastructure like schools and hospitals.
With winter coming and the people in Gaza being forced to live in an ever-smaller section of the strip, attempts to address the humanitarian catastrophe cannot regress, aid must be ramped up. The people of Gaza need aid, food, water, fuel, shelter, and medicine in huge volumes, to ensure hospitals function and lives are saved. We know the risk of disease is high and must be mitigated.
Those displaced in this conflict also need assurances of their right to return home and rebuild their lives. Gaza cannot be left as a refugee camp, there can be no reoccupation or reduction of its territory.
The UK and partners must start work immediately to find a pathway to an enduring cessation of hostilities and a lasting political solution. We want to see the threat of Hamas removed, the end to illegal settlements and settler violence in the West Bank, and a plan for the reconstruction and renewal of Gaza.
Palestinians must be assured their future will not be like the past, that they and their children will be able to enjoy the security, opportunities and rights that we take for granted.
That will not be easy. Diplomatic work never is. But the past few days have shown what diplomacy can do.
These are the essential steps if we are to deliver a two-state solution, with a Palestinian state alongside a safe and secure Israel, the only credible basis for long-term peace.
Military action without this sort of plan cannot succeed.


"""

# Summarize the text
summary = summarizer(text, max_length=50, min_length=10, do_sample=False)

# Print the summary
print(summary[0]['summary_text'])

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.
Token indices sequence length is longer than the specified maximum sequence length for this model (546 > 512). Running this sequence through the model will result in indexing errors


all sides must work for a return to cessation that would allow for the release of more hostages . the levels of death and destruction over the past weeks has been intolerable . there must be full accountability for all actions
